# 04_transformer_improved — Multitask Transformer (Improved)

This notebook trains **one multitask transformer** with:
- a shared multilingual encoder
- three task-specific heads:
  - `queue`
  - `priority`
  - `type`

Improvements over the first version:
1. **Stronger text formatting**: `Subject: ... Body: ...`
2. **Queue class weighting** to help macro-F1 on imbalanced classes
3. **Masked mean pooling** instead of taking only the first token
4. **Label smoothing** for more stable multitask learning
5. **Warmup + gradient accumulation + early stopping**
6. **Confidence analysis** for queue predictions

In [1]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti SUPER


## Configuration

Notes:
- `xlm-roberta-base` is usually stronger on mixed EN/DE data, but heavier.
- If memory is tight, switch to `distilbert-base-multilingual-cased`.

In [2]:
# Stronger multilingual model; if VRAM is tight, use:
# MODEL_NAME = "distilbert-base-multilingual-cased"
MODEL_NAME = "xlm-roberta-base"

MAX_LENGTH = 320
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LABEL_SMOOTHING = 0.05

LOSS_WEIGHTS = {
    "queue": 0.70,
    "priority": 0.15,
    "type": 0.15,
}

ARTIFACTS_DIR = Path("../artifacts/multitask_transformer_improved")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def read_indices(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(line.strip()) for line in f if line.strip()]

In [4]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset["train"].to_pandas()

train_idx = read_indices("data/train_idx.txt")
val_idx = read_indices("data/val_idx.txt")
test_idx = read_indices("data/test_idx.txt")

train_df = df.iloc[train_idx].reset_index(drop=True).copy()
val_df = df.iloc[val_idx].reset_index(drop=True).copy()
test_df = df.iloc[test_idx].reset_index(drop=True).copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (49412, 16)
val: (6176, 16)
test: (6177, 16)


## Text preparation

In [5]:
def normalize_text_col(s):
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def make_text(df):
    subject = normalize_text_col(df["subject"])
    body = normalize_text_col(df["body"])
    # Slightly more explicit formatting often works better than a raw concat
    return ("Subject: " + subject + " [SEP] Body: " + body).str.strip()

for frame in (train_df, val_df, test_df):
    frame["type"] = frame["type"].fillna("Unknown")
    frame["text"] = make_text(frame)

In [6]:
train_df[["text", "queue", "priority", "type"]].head(3)

,text,queue,priority,type
0,Subject: Wesentlicher Sicherheitsvorfall [SEP]...,Technical Support,high,Incident
1,Subject: Account Disruption [SEP] Body: Dear C...,Technical Support,high,Incident
2,Subject: Query About Smart Home System Integra...,Returns and Exchanges,medium,Request


## Label encoding

In [7]:
queue_le = LabelEncoder()
priority_le = LabelEncoder()
type_le = LabelEncoder()

train_df["label_queue"] = queue_le.fit_transform(train_df["queue"])
val_df["label_queue"] = queue_le.transform(val_df["queue"])
test_df["label_queue"] = queue_le.transform(test_df["queue"])

train_df["label_priority"] = priority_le.fit_transform(train_df["priority"])
val_df["label_priority"] = priority_le.transform(val_df["priority"])
test_df["label_priority"] = priority_le.transform(test_df["priority"])

train_df["label_type"] = type_le.fit_transform(train_df["type"])
val_df["label_type"] = type_le.transform(val_df["type"])
test_df["label_type"] = type_le.transform(test_df["type"])

num_labels_queue = len(queue_le.classes_)
num_labels_priority = len(priority_le.classes_)
num_labels_type = len(type_le.classes_)

print("queue classes:", num_labels_queue)
print("priority classes:", num_labels_priority)
print("type classes:", num_labels_type)

queue classes: 52
priority classes: 5
type classes: 5


## Queue class weights

In [8]:
queue_classes = np.arange(num_labels_queue)
queue_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=queue_classes,
    y=train_df["label_queue"].values
).astype(np.float32)

queue_class_weights = torch.tensor(queue_class_weights, dtype=torch.float32)
queue_class_weights

tensor([4.5905, 3.9104, 4.0094, 3.3109, 3.8009, 3.9593, 0.2466, 4.0958, 3.7708,
        3.5194, 0.1609, 3.6547, 3.7411, 3.6547, 4.0958, 4.0094, 1.7861, 3.4181,
        3.4554, 3.6130, 3.7708, 3.3341, 4.3789, 1.3017, 4.3390, 3.8009, 3.3341,
        4.6128, 0.2033, 3.5064, 4.0435, 3.8009, 3.9429, 3.0359, 3.5325, 3.5994,
        4.3192, 3.0852, 3.4181, 0.1331, 5.0814, 0.4858, 0.7738, 4.0782, 4.0094,
        0.6194, 3.4181, 4.2046, 5.0012, 0.0837, 3.8471, 3.5194])

## Hugging Face datasets

In [9]:
train_hf = Dataset.from_pandas(
    train_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False,
)
val_hf = Dataset.from_pandas(
    val_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False,
)
test_hf = Dataset.from_pandas(
    test_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False,
)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_hf.map(tokenize_function, batched=True)
val_tokenized = val_hf.map(tokenize_function, batched=True)
test_tokenized = test_hf.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/49412 [00:00<?, ? examples/s]

Map:   0%|          | 0/6176 [00:00<?, ? examples/s]

Map:   0%|          | 0/6177 [00:00<?, ? examples/s]

## Improved multitask model

Changes vs the earlier version:
- masked mean pooling instead of only first-token pooling
- queue class weights stored as a model buffer

In [11]:
class MultiTaskTransformer(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels_queue,
        num_labels_priority,
        num_labels_type,
        queue_class_weights=None,
        dropout=0.2,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.queue_classifier = nn.Linear(hidden_size, num_labels_queue)
        self.priority_classifier = nn.Linear(hidden_size, num_labels_priority)
        self.type_classifier = nn.Linear(hidden_size, num_labels_type)

        if queue_class_weights is not None:
            self.register_buffer("queue_class_weights", queue_class_weights)
        else:
            self.queue_class_weights = None

    def mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked_hidden = last_hidden_state * mask
        summed = masked_hidden.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        label_queue=None,
        label_priority=None,
        label_type=None,
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = self.mean_pool(outputs.last_hidden_state, attention_mask)
        pooled_output = self.dropout(pooled_output)

        logits_queue = self.queue_classifier(pooled_output)
        logits_priority = self.priority_classifier(pooled_output)
        logits_type = self.type_classifier(pooled_output)

        return {
            "logits_queue": logits_queue,
            "logits_priority": logits_priority,
            "logits_type": logits_type,
        }

## Custom trainer with weighted multitask loss

In [ ]:
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        label_queue = inputs.pop("label_queue")
        label_priority = inputs.pop("label_priority")
        label_type = inputs.pop("label_type")

        outputs = model(**inputs)

        loss_queue = F.cross_entropy(
            outputs["logits_queue"],
            label_queue,
            weight=model.queue_class_weights,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss_priority = F.cross_entropy(
            outputs["logits_priority"],
            label_priority,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss_type = F.cross_entropy(
            outputs["logits_type"],
            label_type,
            label_smoothing=LABEL_SMOOTHING,
        )

        loss = (
            LOSS_WEIGHTS["queue"] * loss_queue
            + LOSS_WEIGHTS["priority"] * loss_priority
            + LOSS_WEIGHTS["type"] * loss_type
        )

        outputs["loss_queue"] = loss_queue.detach()
        outputs["loss_priority"] = loss_priority.detach()
        outputs["loss_type"] = loss_type.detach()

        return (loss, outputs) if return_outputs else loss

## Metrics

In [16]:
def compute_metrics_multitask(eval_pred):
    predictions, labels = eval_pred

    # Приводим predictions к удобному виду
    if isinstance(predictions, dict):
        logits_queue = predictions["logits_queue"]
        logits_priority = predictions["logits_priority"]
        logits_type = predictions["logits_type"]
    elif isinstance(predictions, (list, tuple)):
        # берем первые 3 элемента как logits трех задач
        logits_queue = predictions[0]
        logits_priority = predictions[1]
        logits_type = predictions[2]
    else:
        raise ValueError(f"Unexpected predictions type: {type(predictions)}")

    # labels тоже могут прийти tuple/list
    if isinstance(labels, (list, tuple)):
        labels_queue = labels[0]
        labels_priority = labels[1]
        labels_type = labels[2]
    else:
        raise ValueError(f"Unexpected labels type: {type(labels)}")

    preds_queue = np.argmax(logits_queue, axis=1)
    preds_priority = np.argmax(logits_priority, axis=1)
    preds_type = np.argmax(logits_type, axis=1)

    macro_f1_queue = f1_score(labels_queue, preds_queue, average="macro")
    acc_queue = accuracy_score(labels_queue, preds_queue)
    acc_priority = accuracy_score(labels_priority, preds_priority)
    acc_type = accuracy_score(labels_type, preds_type)

    final_score = (
        0.70 * macro_f1_queue
        + 0.15 * acc_priority
        + 0.15 * acc_type
    )

    return {
        "macro_f1_queue": macro_f1_queue,
        "accuracy_queue": acc_queue,
        "accuracy_priority": acc_priority,
        "accuracy_type": acc_type,
        "final_score": final_score,
    }

## Training setup

In [17]:
model = MultiTaskTransformer(
    model_name=MODEL_NAME,
    num_labels_queue=num_labels_queue,
    num_labels_priority=num_labels_priority,
    num_labels_type=num_labels_type,
    queue_class_weights=queue_class_weights,
)

training_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="final_score",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    label_names=["label_queue", "label_priority", "label_type"],
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_multitask,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Train

In [18]:
train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Macro F1 Queue,Accuracy Queue,Accuracy Priority,Accuracy Type,Final Score
1,2.405119,3.424797,0.763165,0.425518,0.433128,0.832740,0.724096
2,1.737967,3.296552,0.819308,0.502915,0.531574,0.843102,0.779717
3,1.605950,3.229870,0.840041,0.503724,0.558776,0.855246,0.800132


TrainOutput(global_step=9267, training_loss=1.916345519130382, metrics={'train_runtime': 1130.8791, 'train_samples_per_second': 131.08, 'train_steps_per_second': 8.195, 'total_flos': 0.0, 'train_loss': 1.916345519130382, 'epoch': 3.0})

## Validation history

In [19]:
log_df = pd.DataFrame(trainer.state.log_history)
val_log_df = log_df[log_df["eval_final_score"].notna()][[
    "epoch",
    "eval_loss",
    "eval_macro_f1_queue",
    "eval_accuracy_queue",
    "eval_accuracy_priority",
    "eval_accuracy_type",
    "eval_final_score",
]].reset_index(drop=True)

val_log_df

,epoch,eval_loss,eval_macro_f1_queue,eval_accuracy_queue,eval_accuracy_priority,eval_accuracy_type,eval_final_score
0,1.0,3.424797,0.763165,0.425518,0.433128,0.832740,0.724096
1,2.0,3.296552,0.819308,0.502915,0.531574,0.843102,0.779717
2,3.0,3.229870,0.840041,0.503724,0.558776,0.855246,0.800132


## Test set evaluation

In [22]:
test_output = trainer.predict(test_tokenized)

predictions = test_output.predictions
labels = test_output.label_ids

# Извлекаем logits
if isinstance(predictions, dict):
    logits_queue = predictions["logits_queue"]
    logits_priority = predictions["logits_priority"]
    logits_type = predictions["logits_type"]

elif isinstance(predictions, (list, tuple)):
    # берем первые 3 массива как logits трех задач
    logits_queue = predictions[0]
    logits_priority = predictions[1]
    logits_type = predictions[2]

else:
    raise ValueError(f"Unexpected predictions type: {type(predictions)}")

# Извлекаем labels
if isinstance(labels, (list, tuple)):
    labels_queue = labels[0]
    labels_priority = labels[1]
    labels_type = labels[2]
else:
    raise ValueError(f"Unexpected labels type: {type(labels)}")

preds_queue = np.argmax(logits_queue, axis=1)
preds_priority = np.argmax(logits_priority, axis=1)
preds_type = np.argmax(logits_type, axis=1)

test_macro_f1_queue = f1_score(labels_queue, preds_queue, average="macro")
test_acc_queue = accuracy_score(labels_queue, preds_queue)
test_acc_priority = accuracy_score(labels_priority, preds_priority)
test_acc_type = accuracy_score(labels_type, preds_type)

final_score = (
    0.70 * test_macro_f1_queue
    + 0.15 * test_acc_priority
    + 0.15 * test_acc_type
)

print("Test Macro-F1 (queue):", test_macro_f1_queue)
print("Test Accuracy (queue):", test_acc_queue)
print("Test Accuracy (priority):", test_acc_priority)
print("Test Accuracy (type):", test_acc_type)
print("Final Score:", final_score)

Test Macro-F1 (queue): 0.8457214734748246
Test Accuracy (queue): 0.5004047272138579
Test Accuracy (priority): 0.5491338837623442
Test Accuracy (type): 0.8539744212400842
Final Score: 0.8024712771827414


## Confidence analysis for `queue`

In [23]:
probs_queue = torch.softmax(torch.tensor(logits_queue), dim=1).numpy()
conf_queue = probs_queue.max(axis=1)

queue_conf_df = pd.DataFrame({
    "true_queue": queue_le.inverse_transform(labels_queue),
    "pred_queue": queue_le.inverse_transform(preds_queue),
    "confidence": conf_queue,
})

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]
rows = []

for thr in thresholds:
    mask = conf_queue >= thr
    coverage = mask.mean()
    if mask.sum() > 0:
        macro_f1 = f1_score(labels_queue[mask], preds_queue[mask], average="macro")
        acc = accuracy_score(labels_queue[mask], preds_queue[mask])
    else:
        macro_f1 = np.nan
        acc = np.nan

    rows.append({
        "threshold": thr,
        "coverage": coverage,
        "macro_f1_queue": macro_f1,
        "accuracy_queue": acc,
    })

confidence_df = pd.DataFrame(rows)
confidence_df

,threshold,coverage,macro_f1_queue,accuracy_queue
0,0.5,0.293346,0.878095,0.944812
1,0.6,0.283957,0.891293,0.960091
2,0.7,0.231180,0.898877,0.966387
3,0.8,0.213372,0.928491,0.982549
4,0.9,0.190384,0.991953,0.991497


## Save artifacts

In [24]:
pred_df = pd.DataFrame({
    "text": test_df["text"],
    "true_queue": queue_le.inverse_transform(labels_queue),
    "pred_queue": queue_le.inverse_transform(preds_queue),
    "true_priority": priority_le.inverse_transform(labels_priority),
    "pred_priority": priority_le.inverse_transform(preds_priority),
    "true_type": type_le.inverse_transform(labels_type),
    "pred_type": type_le.inverse_transform(preds_type),
    "queue_confidence": conf_queue,
})

summary = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "label_smoothing": LABEL_SMOOTHING,
    "test_macro_f1_queue": float(test_macro_f1_queue),
    "test_accuracy_queue": float(test_acc_queue),
    "test_accuracy_priority": float(test_acc_priority),
    "test_accuracy_type": float(test_acc_type),
    "final_score": float(final_score),
}

pred_df.to_csv(ARTIFACTS_DIR / "test_predictions.csv", index=False)
val_log_df.to_csv(ARTIFACTS_DIR / "validation_history.csv", index=False)
confidence_df.to_csv(ARTIFACTS_DIR / "queue_confidence_analysis.csv", index=False)
pd.DataFrame([summary]).to_csv(ARTIFACTS_DIR / "summary.csv", index=False)

summary

{'model_name': 'xlm-roberta-base',
 'max_length': 320,
 'num_epochs': 3,
 'train_batch_size': 8,
 'eval_batch_size': 16,
 'gradient_accumulation_steps': 2,
 'learning_rate': 2e-05,
 'weight_decay': 0.01,
 'warmup_ratio': 0.1,
 'label_smoothing': 0.05,
 'test_macro_f1_queue': 0.8457214734748246,
 'test_accuracy_queue': 0.5004047272138579,
 'test_accuracy_priority': 0.5491338837623442,
 'test_accuracy_type': 0.8539744212400842,
 'final_score': 0.8024712771827414}

## Final notes for the report

If this improved model beats the previous score, report:
- what changed
- which change likely helped the most
- whether gains came mainly from `queue` or also from `priority` / `type`

Suggested explanation:
- stronger encoder or better pooling improved text representation
- queue class weighting improved rare-class performance
- multitask sharing still preserved good auxiliary-task accuracy